In [ ]:
%sql 
use catalog mobills; 
with receitas as (
    select 
        cal.year_month,
        sum(valor) as vl_receitas
    from gold.transacoes trs 
    left join gold.calendar cal on trs.data = cal.calendar_date
    where trs.natureza = 'Receitas'
    group by cal.year_month
),

despesas as (
    select 
        cal.year_month,
        trs.categoria,
        sum(trs.valor_abs) as valor
    from gold.transacoes trs
    left join gold.calendar cal on trs.data = cal.calendar_date
    where trs.natureza = 'Despesas'
    group by all
)

select 
    d.year_month,
    d.categoria,
    d.valor,
    coalesce(r.vl_receitas, 0) as vl_receitas,
    d.valor / r.vl_receitas as prc_receita
from despesas d 
left join receitas r on d.year_month = r.year_month
order by d.year_month desc, d.categoria

In [ ]:
(
    _sqldf
    .write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable("mobills.gold.monthly_income_share")
)